# 巨大なデータセットをダウンロードせずにストリーミングで読む

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_transformers/datasets_streaming.ipynb)

NLP クックブック 12-11 ／ Transformers の基本操作

Wikipedia 全体のような数 GB〜数百 GB のデータセットから、先頭の一部だけ、あるいは条件に合うものだけを取り出したい。

In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset

wiki = load_dataset("wikimedia/wikipedia", "20231101.ja", split="train", streaming=True)   # 日本語版 Wikipedia (約 140 万記事)
for article in wiki.take(3):
    print(article["title"], "|", len(article["text"]), "文字 |", article["text"][:40].replace("\n", " "))

アンパサンド | 2707 文字 | アンパサンド（&, ）は、並立助詞「…と…」を意味する記号である。ラテン語で「…
言語 | 10788 文字 | 言語（げんご）は、狭義には「声による記号の体系」をいう。  広辞苑や大辞泉には次
日本語 | 69453 文字 | 日本語（にほんご、にっぽんご）は、日本国内や、かつての日本領だった国、そして国外


In [3]:
# 条件に合う記事だけを必要な数だけ集める (全体はダウンロードされない)
nlp_articles = wiki.filter(lambda a: "自然言語処理" in a["text"]).take(5)
print([a["title"] for a in nlp_articles])

# シャッフル (バッファ内での近似的なシャッフル) と変換
sample = wiki.shuffle(seed=0, buffer_size=1000).map(lambda a: {"first": a["text"].split("。")[0]}).take(3)
for a in sample:
    print(a["title"], "→", a["first"][:50])

['自然言語処理', '自然言語', 'プログラミング言語', '人工知能', '検索エンジン']
ベイビー・アリエル → アリエル・レベッカ・マーティン (Ariel Rebecca Martin、2000年11月22日 
パリ → パリ市（パリし、）、通称パリ（、巴里）は、フランスの首都
形式言語 → 形式言語（けいしきげんご、）は、その文法（構文、統語論）が、場合によっては意味（意味論）も、形式的に


- `streaming=True` だと `IterableDataset` になり、`ds[0]` のような添字アクセスや `len()` は使えない。
- `shuffle` はバッファサイズ分しか混ざらない。完全なランダムサンプルが必要なら、ダウンロードして通常のデータセットとして扱う。
- 取り出したものを保存するなら `Dataset.from_list(list(wiki.take(10000)))`。